# AIKO — Qwen3-TTS Colab test

Tests `Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice` on a Colab NVIDIA GPU through **vLLM-Omni**.

Included:
- Japanese TTS
- latency, audio duration, RTF and GPU-memory measurements
- small concurrency benchmark
- **true streamed PCM** test with time-to-first-audio
- live Gradio streaming UI

Recommended runtime: **A100 80 GB** for benchmarking. Smaller NVIDIA GPUs may also work.


In [ ]:
!nvidia-smi
import sys
print(sys.version)


## 1. Install
A runtime restart may be required if Colab changes core CUDA/PyTorch packages.


In [ ]:
%pip install -q -U "vllm-omni" gradio httpx soundfile numpy


## 2. Optional Google Drive output folder


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
OUT_DIR = Path("/content/drive/MyDrive/AIKO/tts-tests/qwen3")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs:", OUT_DIR)


## 3. Start Qwen3-TTS server
The first launch downloads the model weights.


In [ ]:
import os, subprocess, time, httpx, signal, textwrap

MODEL = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice"
PORT = 8091
BASE = f"http://127.0.0.1:{PORT}"

log_path = "/content/qwen_vllm.log"
log = open(log_path, "w")

server = subprocess.Popen(
    [
        "vllm", "serve", MODEL,
        "--omni",
        "--port", str(PORT),
        "--trust-remote-code",
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
    env={**os.environ, "PYTHONUNBUFFERED":"1"},
)

deadline = time.time() + 900
while time.time() < deadline:
    if server.poll() is not None:
        raise RuntimeError(f"Server exited. Check {log_path}")
    try:
        r = httpx.get(f"{BASE}/v1/audio/voices", timeout=5)
        if r.status_code < 500:
            print("Qwen server ready:", r.status_code)
            break
    except Exception:
        pass
    print("Loading Qwen3-TTS...", end="\r")
    time.sleep(5)
else:
    raise TimeoutError(f"Server did not become ready. Check {log_path}")


## 4. Generate one Japanese line


In [ ]:
import io, json, time, subprocess, soundfile as sf
from IPython.display import Audio, display

def gpu_used_mib():
    out = subprocess.check_output([
        "nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"
    ], text=True).strip().splitlines()[0]
    return int(out)

def qwen_tts(text, speaker="Ono_Anna", instructions="", output_name="qwen.wav"):
    payload = {
        "input": text,
        "voice": speaker,
        "language": "Japanese",
        "response_format": "wav",
    }
    if instructions.strip():
        payload["instructions"] = instructions.strip()

    t0 = time.perf_counter()
    r = httpx.post(f"{BASE}/v1/audio/speech", json=payload, timeout=300)
    r.raise_for_status()
    elapsed = time.perf_counter() - t0

    path = OUT_DIR / output_name
    path.write_bytes(r.content)
    data, sr = sf.read(io.BytesIO(r.content))
    duration = len(data) / sr
    metrics = {
        "latency_s": round(elapsed, 3),
        "audio_s": round(duration, 3),
        "rtf": round(elapsed / duration, 3),
        "gpu_used_mib": gpu_used_mib(),
    }
    return path, metrics

TEXT = "今日はいい天気ですね。いっしょに公園へ行きませんか？"
path, metrics = qwen_tts(
    TEXT,
    instructions="自然で親しみやすい会話調。少し明るく、教科書の朗読のようにはしない。",
    output_name="qwen_japanese_test.wav",
)
print(metrics)
display(Audio(filename=str(path)))


## 5. Small AIKO-style benchmark


In [ ]:
PROMPTS = {
    "short_word": "学校",
    "question": "今日は学校で何を勉強しましたか？",
    "natural": "えっ、本当に？それはちょっとびっくりした。でも、きっと大丈夫だよ。",
    "teacher": "この漢字は「閉める」と読みます。窓を閉めてください。",
}

results = []
for name, text in PROMPTS.items():
    path, m = qwen_tts(text, output_name=f"qwen_{name}.wav")
    row = {"case": name, "chars": len(text), **m}
    results.append(row)
    print(row)

print(json.dumps(results, ensure_ascii=False, indent=2))


## 6. True streaming test
This requests raw 24 kHz PCM and measures first received chunk plus first non-silent chunk.


In [ ]:
import numpy as np

def qwen_stream_metrics(text, speaker="Ono_Anna"):
    payload = {
        "input": text,
        "voice": speaker,
        "language": "Japanese",
        "stream": True,
        "stream_format": "audio",
        "response_format": "pcm",
    }
    t0 = time.perf_counter()
    first_chunk = None
    first_audible = None
    chunks = []

    with httpx.stream("POST", f"{BASE}/v1/audio/speech", json=payload, timeout=300) as r:
        r.raise_for_status()
        for chunk in r.iter_bytes():
            if not chunk:
                continue
            now = time.perf_counter()
            if first_chunk is None:
                first_chunk = now - t0
            chunks.append(chunk)
            pcm = np.frombuffer(chunk[:len(chunk)//2*2], dtype=np.int16)
            if first_audible is None and pcm.size and np.max(np.abs(pcm.astype(np.int32))) > 128:
                first_audible = now - t0

    total = time.perf_counter() - t0
    raw = b"".join(chunks)
    wav = np.frombuffer(raw[:len(raw)//2*2], dtype=np.int16)
    duration = len(wav) / 24000
    out = OUT_DIR / "qwen_stream.wav"
    sf.write(out, wav.astype(np.float32) / 32768.0, 24000)

    return out, {
        "first_chunk_s": round(first_chunk or total, 3),
        "first_audible_chunk_s": round(first_audible or total, 3),
        "total_request_s": round(total, 3),
        "audio_s": round(duration, 3),
        "rtf": round(total / duration, 3) if duration else None,
    }

sp, sm = qwen_stream_metrics("ねえ、ちょっと聞いて。今日、面白いことがあったんだ。")
print(sm)
display(Audio(filename=str(sp)))


## 7. Concurrency benchmark
Keep the default small first. Increase only after checking VRAM and stability.


In [ ]:
import asyncio

async def one_request(client, text):
    payload = {
        "input": text,
        "voice": "Ono_Anna",
        "language": "Japanese",
        "response_format": "wav",
    }
    t0 = time.perf_counter()
    r = await client.post(f"{BASE}/v1/audio/speech", json=payload, timeout=300)
    r.raise_for_status()
    return time.perf_counter() - t0

async def bench_concurrency(n):
    async with httpx.AsyncClient() as client:
        t0 = time.perf_counter()
        lat = await asyncio.gather(*[
            one_request(client, f"これは同時リクエスト {i+1} のテストです。")
            for i in range(n)
        ])
        wall = time.perf_counter() - t0
    return {
        "concurrency": n,
        "wall_s": round(wall, 3),
        "avg_latency_s": round(sum(lat)/len(lat), 3),
        "requests_per_s": round(n/wall, 3),
        "gpu_used_mib": gpu_used_mib(),
    }

for n in [1, 2, 4]:
    print(await bench_concurrency(n))


## 8. Live streaming Gradio UI
Open the public Gradio link. Audio is yielded as streamed PCM chunks.


In [ ]:
import gradio as gr

def stream_for_gradio(text, mood):
    if not text.strip():
        return
    payload = {
        "input": text.strip(),
        "voice": "Ono_Anna",
        "language": "Japanese",
        "stream": True,
        "stream_format": "audio",
        "response_format": "pcm",
    }
    if mood.strip():
        payload["instructions"] = mood.strip()

    with httpx.stream("POST", f"{BASE}/v1/audio/speech", json=payload, timeout=300) as r:
        r.raise_for_status()
        for chunk in r.iter_bytes():
            if not chunk:
                continue
            pcm = np.frombuffer(chunk[:len(chunk)//2*2], dtype=np.int16)
            if pcm.size:
                yield 24000, pcm

with gr.Blocks() as demo:
    gr.Markdown("# AIKO — Qwen3-TTS live Japanese")
    text = gr.Textbox(
        value="こんにちは！今日は何を勉強したいですか？",
        label="Japanese text",
        lines=3,
    )
    mood = gr.Textbox(
        value="自然で親しみやすい会話調。明るく。",
        label="Style / emotion",
    )
    go = gr.Button("Speak")
    audio = gr.Audio(label="Streaming audio", streaming=True, autoplay=True)
    go.click(stream_for_gradio, [text, mood], audio)

demo.queue().launch(share=True, debug=False)


## 9. Stop server


In [ ]:
server.terminate()
server.wait(timeout=30)
log.close()
print("Stopped.")
